# Sample Review — Structured JSON

Side-by-side comparison of structured JSON output from both models.

| Colour | Label |
|---|---|
| 🔴 Red header | Document title |
| 🔵 Blue | Section title |
| Dark + blue border | Section description |
| 🟡 Yellow | Question text |
| Dark + green border | Answer text |

In [1]:
from pathlib import Path
import json
from html import escape
from IPython.display import HTML, display

ROOT    = Path("..") if (Path("..") / "data").exists() else Path(".")
LLM_DIR = ROOT / "data" / "llmlabeled"

MODELS = ["llama3.3-70b", "llama3.1-8b"]
MODEL_COLORS = {
    "llama3.3-70b": "#7048e8",
    "llama3.1-8b":  "#1098ad",
}

SAMPLES = sorted(
    [p.stem.replace("_llama3.3-70b_structured", "")
     for p in LLM_DIR.glob("*_llama3.3-70b_structured.json")],
    key=lambda s: int(s.replace("sample", ""))
)

print(f"Found {len(SAMPLES)} samples: {SAMPLES}")
print(f"Models: {MODELS}")

Found 10 samples: ['sample1', 'sample2', 'sample3', 'sample4', 'sample5', 'sample6', 'sample7', 'sample8', 'sample9', 'sample10']
Models: ['llama3.3-70b', 'llama3.1-8b']


In [2]:
def render_structured(data: dict, model_tag: str) -> str:
    template = data["narrative"]["template"]
    color    = MODEL_COLORS.get(model_tag, "#555")
    parts    = []

    # Model badge
    parts.append(
        f'<div style="background:{color};color:white;padding:7px 12px;'
        f'font-size:12px;font-weight:bold;border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;margin-bottom:10px;">'
        f'{model_tag}</div>'
    )

    # Title
    title = template.get("title", "").strip()
    if title:
        parts.append(
            f'<div style="background:#ff6b6b;color:white;padding:10px 12px;'
            f'font-size:15px;font-weight:bold;border-radius:6px;margin-bottom:10px;">'
            f'{escape(title)}</div>'
        )

    # Template description
    desc = template.get("description", "").strip()
    if desc:
        parts.append(
            f'<div style="background:#161b22;color:#c9d1d9;padding:10px 12px;'
            f'margin-bottom:10px;border-left:5px solid #ff6b6b;'
            f'white-space:pre-wrap;border-radius:4px;font-size:13px;">'
            f'{escape(desc)}</div>'
        )

    for section in template.get("section", []):
        sec_title = section.get("title", "").strip()
        parts.append(
            f'<div style="background:#4dabf7;color:white;padding:8px 12px;'
            f'font-weight:bold;border-radius:6px;margin-top:12px;font-size:13px;">'
            f'{escape(sec_title)}</div>'
        )

        sec_desc = section.get("description", "").strip()
        if sec_desc:
            parts.append(
                f'<div style="background:#161b22;color:#c9d1d9;padding:10px 12px;'
                f'margin-bottom:8px;border-left:5px solid #4dabf7;'
                f'white-space:pre-wrap;border-radius:4px;font-size:12px;">'
                f'{escape(sec_desc)}</div>'
            )

        for question in section.get("question", []):
            q_text = question.get("text", "").strip()
            answer = question.get("answer", {}).get("json", {}).get("answer", "")

            if q_text:
                parts.append(
                    f'<div style="background:#ffd43b;color:black;padding:7px 10px;'
                    f'margin-top:8px;border-radius:4px;font-weight:bold;font-size:12px;">'
                    f'{escape(q_text)}</div>'
                )

            parts.append(
                f'<div style="background:#161b22;color:#e6edf3;padding:10px 12px;'
                f'margin-bottom:6px;border-left:5px solid #51cf66;'
                f'white-space:pre-wrap;border-radius:4px;font-size:12px;">'
                f'{escape(answer)}</div>'
            )

    return "".join(parts)

In [3]:
# Change SAMPLE to a specific name (e.g. "sample3") or keep "all"
SAMPLE = "all"

to_show = SAMPLES if SAMPLE == "all" else [SAMPLE]

for sample in to_show:
    cols = []
    for model in MODELS:
        path = LLM_DIR / f"{sample}_{model}_structured.json"
        if path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            cols.append(render_structured(data, model))
        else:
            cols.append(
                f'<div style="color:#888;padding:20px;">Not found: {path.name}</div>'
            )

    display(HTML(
        f'<details open>'
        f'<summary style="font-size:17px;font-weight:bold;padding:10px;cursor:pointer;">'
        f'{sample}</summary>'
        f'<div style="display:flex;gap:12px;background:#0d1117;padding:16px;'
        f'border-radius:8px;max-height:900px;overflow:auto;">'
        f'<div style="flex:1;min-width:0;">{cols[0]}</div>'
        f'<div style="width:1px;background:#30363d;"></div>'
        f'<div style="flex:1;min-width:0;">{cols[1]}</div>'
        f'</div></details>'
    ))